In [2]:
import sys, pandas
print(sys.executable)
print(pandas.__version__)

d:\Cyber Security\Research\J26-CS-336\.venv\Scripts\python.exe
3.0.6


In [3]:
from pathlib import Path
import pandas as pd

RAW = Path("../../data/raw/cicapt_provenance")
for f in RAW.iterdir():
    print(f.name)

Phase1_Provenance.csv
Phase2_Provenance.csv


In [4]:
p1 = pd.read_csv(RAW / "Phase1_Provenance.csv", dtype=str)
p2 = pd.read_csv(RAW / "Phase2_Provenance.csv", dtype=str)
print("Phase 1:", p1.shape)
print("Phase 2:", p2.shape)

Phase 1: (222120, 35)
Phase 2: (196735, 35)


In [5]:
p2.head(10)

,id,type,from,to,uid,egid,exe,gid,euid,name,...,local address,remote port,protocol,remote address,local port,tgid,fd,mode,label,subLabel
0,ca5b322fa4d4cb63aa5dd9fbd88e37e4,Process,NaN,NaN,0.0,0.0,/usr/bin/dash,0.0,0.0,ethtool,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
1,e36715b62c8cdc32e47483b3600712f1,Process,NaN,NaN,0.0,0.0,/usr/bin/dash,0.0,0.0,ethtool,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
2,NaN,WasTriggeredBy,e36715b62c8cdc32e47483b3600712f1,ca5b322fa4d4cb63aa5dd9fbd88e37e4,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
3,216f4d4893942a66d649c4e02b5722c4,Artifact,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
4,NaN,Used,e36715b62c8cdc32e47483b3600712f1,216f4d4893942a66d649c4e02b5722c4,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
5,bdc2ca1d94be5a3958f6621d8b52fdb0,Artifact,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
6,NaN,Used,e36715b62c8cdc32e47483b3600712f1,bdc2ca1d94be5a3958f6621d8b52fdb0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
7,02a4195a9866b7c9eba7405c13110dad,Artifact,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
8,NaN,Used,e36715b62c8cdc32e47483b3600712f1,02a4195a9866b7c9eba7405c13110dad,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
9,b55dcc91b1270f16cef024e045a3aa76,Process,NaN,NaN,0.0,0.0,/usr/bin/dash,0.0,0.0,openvpn,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0


In [6]:
for name, df in [("Phase 1", p1), ("Phase 2", p2)]:
    print(f"\n===== {name}")
    print(df["type"].value_counts())


===== Phase 1
type
WasGeneratedBy    136444
Artifact           36895
Used               22120
WasDerivedFrom     11031
Process             9878
WasTriggeredBy      5752
Name: count, dtype: int64

===== Phase 2
type
WasGeneratedBy    87636
Used              34206
Artifact          28447
Process           24839
WasTriggeredBy    13136
WasDerivedFrom     8471
Name: count, dtype: int64


In [7]:
for name, df in [("Phase 1", p1), ("Phase 2", p2)]:
    print(f"\n===== {name}")
    print(df["label"].value_counts())
    print(df["subLabel"].value_counts())


===== Phase 1
label
0    222120
Name: count, dtype: int64
subLabel
0    222120
Name: count, dtype: int64

===== Phase 2
label
0    196333
1       402
Name: count, dtype: int64
subLabel
0                   196333
credentialAccess       151
collection             100
defenceEvasion          46
discovery               36
exfiltration            22
persistence             19
CandC                   16
lateralMovement         10
initialAccess            2
Name: count, dtype: int64


In [8]:
time_cols = ["time", "start time", "seen time"]

for name, df in [("Phase 1", p1), ("Phase 2", p2)]:
    for c in time_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    t = df[time_cols].min(axis=1)
    print(f"{name}: {pd.to_datetime(t.min(), unit='s')}  ->  {pd.to_datetime(t.max(), unit='s')}"
          f"   ({(t.max() - t.min()) / 3600:.1f} hours)")

Phase 1: 2023-11-27 20:50:10.496000051  ->  2023-12-01 20:50:13.812999964   (96.0 hours)
Phase 2: 2023-12-01 22:13:45.766999960  ->  2023-12-04 22:20:21.461999893   (72.1 hours)


In [9]:
hours = {"Phase 1": 96.0, "Phase 2": 72.1}
rates = {}
for name, df in [("Phase 1", p1), ("Phase 2", p2)]:
    procs = df[df["type"] == "Process"]
    rates[name] = procs["name"].value_counts() / hours[name]

compare = pd.DataFrame(rates).fillna(0).round(1)
compare["difference"] = compare["Phase 2"] - compare["Phase 1"]
compare.sort_values("difference", ascending=False).head(15)

,Phase 1,Phase 2,difference
name,,,
cp,0.3,195.6,195.3
apt-config,4.7,11.9,7.2
dpkg,6.6,13.5,6.9
lsb_release,3.1,7.0,3.9
cmp,1.3,4.9,3.6
cat,1.4,4.4,3.0
sh,3.6,6.5,2.9
gpg-connect-age,0.5,3.0,2.5
update-alternat,0.0,1.9,1.9


In [10]:
procs2 = p2[p2["type"] == "Process"].copy()
cp = procs2[procs2["name"] == "cp"]

print("cp processes:", len(cp))
print(cp["label"].value_counts())

print("\nTop parent PIDs of cp:")
print(cp["ppid"].value_counts().head(5))

staging = cp["command line"].fillna("").str.contains(r"staged/\.s|/tmp/\.s")
print("\ncp commands writing to the attacker's staging folders:", staging.sum())
print(cp.loc[staging, "label"].value_counts())

cp processes: 14102
label
0    14090
1       12
Name: count, dtype: int64

Top parent PIDs of cp:
ppid
64561.0    8898
59438.0    5108
46623.0       8
2585.0        2
2668.0        2
Name: count, dtype: int64

cp commands writing to the attacker's staging folders: 7008
label
0    7005
1       3
Name: count, dtype: int64


In [11]:
procs2["t"] = procs2["start time"].fillna(procs2["seen time"])
procs2["pid_n"] = pd.to_numeric(procs2["pid"])
procs2["ppid_n"] = pd.to_numeric(procs2["ppid"])

# 1. Group the labelled attack processes into attack steps (a gap of more than 5 min = new step)
att = procs2[procs2["label"] == "1"].sort_values("t").copy()
att["burst"] = (att["t"].diff() > 300).cumsum()
bursts = att.groupby("burst").agg(t0=("t", "min"), pid_min=("pid_n", "min"),
                                  pid_max=("pid_n", "max"), tactic=("subLabel", "first"))
print("attack steps:", len(bursts))

# 2. Start from the original labels
procs2["label_fixed"] = procs2["label"].eq("1")
procs2["tactic_fixed"] = procs2["subLabel"]
known_pids = set(procs2["pid_n"])

# 3. Rule 2: missing parent inside an attack step's PID block, started within 1 hour
for _, b in bursts.iterrows():
    hit = (~procs2["label_fixed"]
           & ~procs2["ppid_n"].isin(known_pids)
           & procs2["ppid_n"].between(b.pid_min, b.pid_max)
           & procs2["t"].between(b.t0, b.t0 + 3600))
    procs2.loc[hit, "label_fixed"] = True
    procs2.loc[hit, "tactic_fixed"] = b.tactic

# 4. Rule 1: children of attack processes, repeated until nothing changes
while True:
    att_pids = set(procs2.loc[procs2["label_fixed"], "pid_n"])
    hit = ~procs2["label_fixed"] & procs2["ppid_n"].isin(att_pids)
    if not hit.any():
        break
    procs2.loc[hit, "label_fixed"] = True

changed = procs2[procs2["label_fixed"] & (procs2["label"] == "0")]
print("relabelled as attack:", len(changed))
print(changed["name"].value_counts())
print("attack process nodes before:", (procs2["label"] == "1").sum(),
      " after:", procs2["label_fixed"].sum(), " of", len(procs2))

attack steps: 47
relabelled as attack: 14054
name
cp        14010
base64       42
grep          2
Name: count, dtype: int64
attack process nodes before: 213  after: 14267  of 24839


In [12]:
OUT = Path("../../data/interim/device_telemetry")
OUT.mkdir(parents=True, exist_ok=True)

procs2.to_parquet(OUT / "phase2_processes_labelfixed.parquet", index=False)
print("saved:", len(procs2), "rows")

saved: 24839 rows


In [13]:
import networkx as nx

# Links between nodes of the same process: WasTriggeredBy edges where both ends have the same PID
wtb = p2[p2["type"] == "WasTriggeredBy"][["from", "to", "operation"]]
pid_of = dict(zip(procs2["id"], procs2["pid"]))
wtb = wtb[wtb["from"].map(pid_of) == wtb["to"].map(pid_of)]
print("same-process links:", len(wtb))
print(wtb["operation"].value_counts())

# Each connected group of nodes = one real process
G = nx.Graph()
G.add_nodes_from(procs2["id"])
G.add_edges_from(zip(wtb["from"], wtb["to"]))
group_of = {node: i for i, group in enumerate(nx.connected_components(G)) for node in group}
procs2["proc_key"] = procs2["id"].map(group_of)

per_proc = procs2.groupby("proc_key")
print("\nreal processes:", per_proc.ngroups)
print("nodes per process:", per_proc.size().value_counts().sort_index().to_dict())

labels = per_proc.agg(orig=("label", lambda s: (s == "1").any()), fixed=("label_fixed", "max"))
print("attack processes, original labels:", labels["orig"].sum())
print("attack processes, fixed labels:  ", labels["fixed"].sum())
print("processes with mixed labels:", (per_proc["label_fixed"].nunique() > 1).sum())

same-process links: 13136
operation
execve    12223
setgid      512
setuid      293
update      108
Name: count, dtype: int64

real processes: 12443
nodes per process: {1: 260, 2: 12059, 3: 80, 4: 9, 5: 25, 6: 10}
attack processes, original labels: 110
attack processes, fixed labels:   7137
processes with mixed labels: 0


In [14]:
procs2["tactic_nz"] = procs2["tactic_fixed"].where(procs2["tactic_fixed"] != "0")

def first_value(s):
    s = s.dropna()
    return s.iloc[0] if len(s) else None

proc_table = (procs2.sort_values("t")
    .groupby("proc_key")
    .agg(pid=("pid_n", "first"),
         ppid=("ppid_n", "first"),
         t_start=("t", "min"),
         t_end=("t", "max"),
         name=("name", "last"),
         exe=("exe", "last"),
         cmdline=("command line", first_value),
         uid=("uid", "first"),
         euid=("euid", "first"),
         n_nodes=("id", "size"),
         label=("label_fixed", "max"),
         tactic=("tactic_nz", first_value)))

proc_table["tactic"] = proc_table["tactic"].fillna("benign")
print(proc_table.shape)
print(proc_table["tactic"].value_counts())
print("\nmissing values:")
print(proc_table.isna().sum()[proc_table.isna().sum() > 0])

(12443, 12)
tactic
collection          7051
benign              5306
defenceEvasion        22
discovery             19
exfiltration          14
CandC                 10
credentialAccess       9
lateralMovement        6
persistence            5
initialAccess          1
Name: count, dtype: int64

missing values:
cmdline    263
dtype: int64


In [15]:
proc_table.to_parquet(OUT / "phase2_process_table.parquet")

In [16]:
edges2 = p2[p2["from"].notna()].copy()
edges2 = edges2[edges2["type"] != "WasDerivedFrom"]

# node id -> process id
key_of = dict(zip(procs2["id"], procs2["proc_key"]))
proc_ids = set(key_of)

# Which end is the process, and which end is the artifact
edges2["proc_key"] = edges2["from"].map(key_of).fillna(edges2["to"].map(key_of))
edges2["artifact_id"] = edges2["to"].where(edges2["from"].isin(proc_ids), edges2["from"])
edges2.loc[edges2["type"] == "WasTriggeredBy", "artifact_id"] = None

# Add details about the artifact
arts2 = p2[p2["type"] == "Artifact"].set_index("id")
for col in ["subtype", "path", "remote address", "remote port"]:
    edges2[col] = edges2["artifact_id"].map(arts2[col])

events2 = edges2[["proc_key", "time", "type", "operation",
                  "subtype", "path", "remote address", "remote port"]]

print("events:", len(events2), "  without a process:", events2["proc_key"].isna().sum())
print(events2["operation"].value_counts())
print(events2["subtype"].value_counts(dropna=False))
print("processes with events:", events2["proc_key"].nunique(),
      "  median events per process:", events2.groupby("proc_key").size().median())

events: 134978   without a process: 0
operation
unlink            68351
load              25512
execve            12223
connect           10787
rename (read)      8466
rename (write)     8466
setgid              512
setuid              293
open                234
update              108
chmod                26
Name: count, dtype: int64
subtype
link              64088
file              46923
NaN               13136
network socket    10787
directory            26
unknown              18
Name: count, dtype: int64
processes with events: 12443   median events per process: 3.0


In [17]:
events2.to_parquet(OUT / "phase2_events.parquet", index=False)